In [21]:
import pandas as pd
import os



'/home/jupyter/cpsc368_group19/cleaned_schema'

In [22]:
tomatoes = pd.read_csv("../data/tomatoes_clean.csv")
oscars = pd.read_csv("../data/oscars_clean.csv")
letterboxd = pd.read_csv("../data/letterboxd_clean.csv")

In [40]:
movies_df = tomatoes[['title', 'releaseDateTheaters', 'genre', 'originalLanguage']].copy()

movies_df = movies_df.rename(columns={
    'releaseDateTheaters': 'releaseDate',
    'originalLanguage': 'language'
})

movies_df['language'] = movies_df['language'].replace({
    'English (Australia)': 'English',
    'English (United Kingdom)': 'English',
    'English (United States)': 'English',
    'French (Canada)': 'French',
    'French (France)': 'French',
    'Spanish (Spain)': 'Spanish',
    'Spanish (Mexico)': 'Spanish',
    'Portuguese (Brazil)': 'Portuguese'
})

movies_df = movies_df.drop_duplicates(subset=['title', 'releaseDate', 'language']).reset_index(drop=True)
movies_df['movie_id'] = range(1, len(movies_df) + 1)
movies_df = movies_df[['movie_id', 'title', 'releaseDate', 'genre', 'language']]

movies_df.head()

,movie_id,title,releaseDate,genre,language
0,1,Endless Poetry,2017-07-14,Biography,Spanish
1,2,Elephants,2018-12-07,"Romance, Drama",English
2,3,West Side Story,2021-12-10,"Musical, Romance",English
3,4,The Edge of Democracy,2019-06-19,"Documentary, History, Drama",Portuguese
4,5,Tiger 24,2022-06-10,Documentary,English


In [41]:
movie_lookup = movies_df[['movie_id', 'title', 'language']]

In [27]:
language_map = {
    'en': 'English',
    'fr': 'French',
    'es': 'Spanish',
    'de': 'German',
    'it': 'Italian',
    'pt': 'Portuguese',
    'ko': 'Korean',
    'ja': 'Japanese',
    'zh': 'Chinese',
    'ru': 'Russian',
    'hi': 'Hindi',
    'ar': 'Arabic',
    'sv': 'Swedish',
    'no': 'Norwegian',
    'da': 'Danish',
    'nl': 'Dutch',
    'pl': 'Polish',
    'tr': 'Turkish',
    'th': 'Thai',
    'vi': 'Vietnamese',
    'he': 'Hebrew',
    'el': 'Greek'
}

In [42]:
letterboxd_metrics['language'] = letterboxd_metrics['language'].replace(language_map)

In [52]:
rt_metrics = tomatoes[['title', 'releaseDateTheaters', 'originalLanguage', 'audienceScore', 'tomatoMeter']].copy()

rt_metrics = rt_metrics.rename(columns={
    'releaseDateTheaters': 'releaseDate',
    'originalLanguage': 'language'
})

rt_metrics['language'] = rt_metrics['language'].replace({
    'English (Australia)': 'English',
    'English (United Kingdom)': 'English',
    'English (United States)': 'English',
    'French (Canada)': 'French',
    'French (France)': 'French',
    'Spanish (Spain)': 'Spanish',
    'Spanish (Mexico)': 'Spanish',
    'Portuguese (Brazil)': 'Portuguese'
})

rt_metrics['title'] = rt_metrics['title'].str.strip()
rt_metrics['releaseDate'] = rt_metrics['releaseDate'].astype(str).str[:10]
rt_metrics['language'] = rt_metrics['language'].str.strip()

rt_metrics = rt_metrics.merge(
    movies_df[['movie_id', 'title', 'releaseDate', 'language']],
    on=['title', 'releaseDate', 'language'],
    how='left'
)

rt_metrics = rt_metrics[['movie_id', 'audienceScore', 'tomatoMeter']].drop_duplicates(subset=['movie_id'])

rt_metrics.head()
len(rt_metrics)

1608

In [54]:
lb_metrics = letterboxd[['movie_title', 'original_language', 'popularity', 'vote_average', 'vote_count']].copy()

lb_metrics = lb_metrics.rename(columns={
    'movie_title': 'title',
    'original_language': 'language'
})

lb_metrics['language'] = lb_metrics['language'].replace(language_map)
lb_metrics['title'] = lb_metrics['title'].str.strip()
lb_metrics['language'] = lb_metrics['language'].str.strip()

lb_metrics = lb_metrics.merge(
    movies_df[['movie_id', 'title', 'language']],
    on=['title', 'language'],
    how='left'
)

lb_metrics = lb_metrics[['movie_id', 'popularity', 'vote_average', 'vote_count']].drop_duplicates(subset=['movie_id'])

lb_metrics.head()
len(lb_metrics)

1035

In [55]:
metrics_df = rt_metrics.merge(
    lb_metrics,
    on='movie_id',
    how='inner'
)

metrics_df = metrics_df.dropna(subset=['movie_id']).reset_index(drop=True)

metrics_df.head()
len(metrics_df)

1034

In [58]:
metrics_df = metrics_df[['movie_id', 'audienceScore', 'tomatoMeter', 'popularity', 'vote_average', 'vote_count']]
metrics_df.head()

,movie_id,audienceScore,tomatoMeter,popularity,vote_average,vote_count
0,1,80.0,93.0,8.424,7.6,136.0
1,2,95.0,83.0,1.018,6.5,2.0
2,3,93.0,91.0,12.619,0.0,0.0
3,4,68.0,97.0,6.595,7.8,287.0
4,6,67.0,91.0,11.607,6.4,290.0


In [65]:
movies_lookup = movies_df.copy()
movies_lookup['releaseYear'] = pd.to_datetime(movies_lookup['releaseDate'], errors='coerce').dt.year

movies_lookup = movies_lookup[['movie_id', 'title', 'releaseYear']]


oscars_df = oscars[['film', 'year_film', 'year_ceremony', 'ceremony', 'canon_category', 'name', 'winner']].copy()

oscars_df = oscars_df.rename(columns={
    'film': 'title'
})

oscars_df['title'] = oscars_df['title'].str.strip()
oscars_df['year_film'] = pd.to_numeric(oscars_df['year_film'], errors='coerce')

oscars_df.head()

,title,year_film,year_ceremony,ceremony,canon_category,name,winner
0,The Martian,2015,2016,88,ACTOR IN A LEADING ROLE,Matt Damon,False
1,Spotlight,2015,2016,88,ACTOR IN A SUPPORTING ROLE,Mark Ruffalo,False
2,Bridge of Spies,2015,2016,88,ACTOR IN A SUPPORTING ROLE,Mark Rylance,True
3,Creed,2015,2016,88,ACTOR IN A SUPPORTING ROLE,Sylvester Stallone,False
4,Carol,2015,2016,88,ACTRESS IN A LEADING ROLE,Cate Blanchett,False


In [66]:
oscars_merged = oscars_df.merge(
    movies_lookup,
    on='title',
    how='left'
)

In [67]:
oscars_merged['year_diff'] = (
    oscars_merged['year_film'] - oscars_merged['releaseYear']
).abs()

In [79]:
oscars_merged = oscars_merged.sort_values('year_diff')

oscars_merged = oscars_merged.drop_duplicates(
    subset=['title', 'year_film', 'year_ceremony', 'ceremony', 'canon_category', 'name', 'winner'],
    keep='first'
).reset_index(drop=True)

oscars_merged['nomination_id'] = range(1, len(oscars_merged) + 1)

oscars_df = oscars_merged[[
    'nomination_id',
    'movie_id',
    'year_film',
    'year_ceremony',
    'ceremony',
    'canon_category',
    'name',
    'winner'
]].copy()

oscars_df.head()


,nomination_id,movie_id,year_film,year_ceremony,ceremony,canon_category,name,winner
0,1,737,2019,2020,92,WRITING (Original Screenplay),Rian Johnson,False
1,2,43,2020,2021,93,INTERNATIONAL FEATURE FILM,Denmark,True
2,3,1321,2020,2021,93,FILM EDITING,Mikkel E. G. Nielsen,True
3,4,58,2020,2021,93,DOCUMENTARY (Feature),Garrett Bradley/Lauren Domino/Kellen Quinn,False
4,5,795,2020,2021,93,DOCUMENTARY (Feature),Maite Alberdi/Marcela Santibáñez,False


In [82]:
movies_df[movies_df['title'] == 'The Martian']

,movie_id,title,releaseDate,genre,language
607,608,The Martian,2015-10-02,"Sci-fi, Adventure, Drama",English


In [84]:
oscars_df[oscars_df['movie_id'] ==608]

,nomination_id,movie_id,year_film,year_ceremony,ceremony,canon_category,name,winner
157,158,608,2015,2016,88,ACTOR IN A LEADING ROLE,Matt Damon,False
350,351,608,2015,2016,88,WRITING (Adapted Screenplay),Drew Goddard,False
353,354,608,2015,2016,88,VISUAL EFFECTS,Richard Stammers/Anders Langlands/Chris Lawren...,False
357,358,608,2015,2016,88,SOUND MIXING,Paul Massey/Mark Taylor/Mac Ruth,False
362,363,608,2015,2016,88,SOUND EDITING,Oliver Tarney,False
364,365,608,2015,2016,88,ART DIRECTION,Arthur Max/Celia Bobak,False
368,369,608,2015,2016,88,BEST PICTURE,Simon Kinberg/Ridley Scott/Michael Schaefer/Ma...,False


In [86]:
movies_df.to_csv("cpsc368_group19/generated/movies_table.csv", index=False)
metrics_df.to_csv("cpsc368_group19/generated/movie_metrics_table.csv", index=False)
oscars_df.to_csv("cpsc368_group19/generated/oscar_nominations_table.csv", index=False)